# HMI Template (Colbún) — UI/Backend estándar (PRO173/PRO174 style)

Plantilla base para nuevos procedimientos (PROxxx) con el **mismo UI y backend**:
- ZONA EDITABLE (`NODOS`) arriba
- Motor HMI estable (ipywidgets)
- Checklist = acciones
- Regla global “si aplica” **no obligatorio**
- Botones: **SÍ / NO / (opcional STOP)** en la misma fila
- **Volver al paso anterior**
- **Exportar JSON**
- Tablas finales (solo en `END_OK` si `show_tables=True`): **Inputs + Decisiones (solo rombos)**

> En Colab, ejecuta la celda 1 (widgets) y luego la celda 2 (HMI).

In [1]:
# --- COLAB: habilitar ipywidgets (si estás en JupyterLab normalmente no hace falta) ---
!pip -q install ipywidgets
from google.colab import output
output.enable_custom_widget_manager()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.5 MB/s eta 0:00:00


In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import datetime, json, uuid

# ============================================================
# CONFIGURACIÓN RÁPIDA
# ============================================================

PRO_ID = "PRO183"
PRO_TITULO = "Devolución a proveedores"
ENABLE_STOP = True  # <- cámbialo a False si NO quieres botón STOP en este PRO

EN_CURSO = "EN_CURSO"
BLOQUEADO = "BLOQUEADO"
DETENIDO_STOP = "DETENIDO_STOP"
FINALIZADO = "FINALIZADO"

def _now_iso():
    return datetime.datetime.now().isoformat(timespec="seconds")

# ============================================================
# ZONA EDITABLE (AQUÍ SOLO EDITAS NODOS / TEXTOS / CHECKLIST / INPUTS)
# ============================================================

def _es_opcional_por_si_aplica(texto: str) -> bool:
    """Si el texto contiene 'si aplica' en cualquier posición, no es obligatorio."""
    return "si aplica" in (texto or "").lower()

# Motivos STOP (si ENABLE_STOP=True)
MOTIVOS_STOP = [
    "Condición insegura detectada",
    "Falta de información crítica",
    "Sistema no disponible",
    "No se logra contacto",
    "Otro",
]


NODOS = {
    "D1_inmediata": {
        "type": "decision",
        "titulo": "Decisión inicial — Gestión inmediata de devolución",
        "rol": "Especialista de almacenamiento (solicitante)",
        "descripcion": "Una vez haya existido la recepción física y revisión de la documentación, se decide la gestión de la devolución",
        "pregunta": "¿La devolución puede gestionarse de manera inmediata?",
        "opciones": [
            {"label": "SÍ", "next": "T1_migo"},
            {"label": "NO", "next": "T6_recepcion_materiales"},
        ],
        "ayuda": "Seleccione la ruta que corresponda según la condición detectada.",
    },

    "T1_migo": {
        "type": "task",
        "titulo": "Ingresar recepción (MIGO) adjunta documentación",
        "rol": "Especialista de almacenamiento (solicitante)",
        "descripcion": "Registrar la recepción en MIGO y adjuntar la documentación disponible para respaldar la devolución inmediata.",
        "checklist": [
            "Registrar la recepción en MIGO",
            "Adjuntar documentación de respaldo",
            "Verificar que la información de recepción sea consistente",
        ],
        "inputs": [
            {"key":"numero_migo", "label":"Ingresa número de recepción MIGO", "required": True},
        ],
        "validacion": "¿Se registró la recepción y se adjuntó la documentación?",
        "next": "T2_generar_devolucion",
    },

    "T2_generar_devolucion": {
        "type": "task",
        "titulo": "Generar devolución a proveedor",
        "rol": "Especialista de almacenamiento (solicitante)",
        "descripcion": "Generar la devolución al proveedor utilizando la referencia disponible para el caso inmediato.",
        "checklist": [
            "Generar la devolución a proveedor",
            "Verificar los datos de la devolución antes de continuar",
        ],
        "inputs": [],
        "validacion": "¿Se generó la devolución a proveedor?",
        "next": "T3_devolver_proveedor",
    },

    "T3_devolver_proveedor": {
        "type": "task",
        "titulo": "Devolver al proveedor",
        "rol": "Especialista de almacenamiento (solicitante)",
        "descripcion": "Gestionar la devolución física al proveedor. Al emitir la guía de despacho, ingresar los datos de transporte requeridos.",
        "checklist": [
            "Emitir o preparar la guía de despacho según corresponda",
            "Ingresar RUT transportista",
            "Ingresar patente",
            "Ingresar RUT chofer",
            "Ingresar nombre de chofer",
            "Validar la entrega al proveedor",
        ],
        "inputs": [
            {"key":"rut_transportista", "label":"Ingresa RUT transportista", "required": True},
            {"key":"patente", "label":"Ingresa patente", "required": True},
            {"key":"rut_chofer", "label":"Ingresa RUT chofer", "required": True},
            {"key":"nombre_chofer", "label":"Ingresa nombre de chofer", "required": True},
        ],
        "validacion": "¿Se gestionó la devolución al proveedor?",
        "next": "T4_notificar_abast_dev",
    },

    "T4_notificar_abast_dev": {
        "type": "task",
        "titulo": "Notificar abastecimiento de devolución",
        "rol": "Especialista de almacenamiento (solicitante)",
        "descripcion": "Informar a abastecimiento que la devolución fue gestionada para continuar con el seguimiento correspondiente.",
        "checklist": [
            "Notificar a abastecimiento la devolución realizada",
            "Adjuntar o informar antecedentes de respaldo (si aplica)",
        ],
        "inputs": [],
        "validacion": "¿Se notificó a abastecimiento?",
        "next": "T5_notificar_causa",
    },

    "T5_notificar_causa": {
        "type": "task",
        "titulo": "Notificar a proveedor causa de devolución",
        "rol": "Ingeniero de abastecimiento",
        "descripcion": "Comunicar al proveedor la causa de la devolución realizada para dejar trazabilidad del motivo.",
        "checklist": [
            "Informar al proveedor la causa de devolución",
            "Confirmar que el mensaje fue enviado",
        ],
        "inputs": [],
        "validacion": "¿Se notificó al proveedor la causa de la devolución?",
        "next": "END_RAMA_SI",
    },

    "T6_recepcion_materiales": {
        "type": "task",
        "titulo": "Recepción de materiales",
        "rol": "Especialista de almacenamiento (solicitante)",
        "descripcion": "Recibir los materiales para continuar con la revisión del caso cuando la devolución no puede gestionarse de manera inmediata.",
        "checklist": [
            "Recibir los materiales",
            "Dejar registro de la recepción",
        ],
        "inputs": [],
        "validacion": "¿Se recepcionaron los materiales?",
        "next": "T7_revision_tecnica",
    },

    "T7_revision_tecnica": {
        "type": "task",
        "titulo": "Revisión técnica declara requerir devolución",
        "rol": "Especialista de almacenamiento (solicitante)",
        "descripcion": "Realizar la revisión técnica del material y declarar que corresponde gestionar una devolución al proveedor.",
        "checklist": [
            "Realizar revisión técnica",
            "Declarar que el material requiere devolución",
        ],
        "inputs": [],
        "validacion": "¿La revisión técnica declaró requerir devolución?",
        "next": "T8_notificar_compra",
    },

    "T8_notificar_compra": {
        "type": "task",
        "titulo": "Notificar compra necesidad de devolución",
        "rol": "Especialista de almacenamiento (solicitante)",
        "descripcion": "Informar a compras la necesidad de devolución detectada tras la revisión técnica.",
        "checklist": [
            "Notificar a compras la necesidad de devolución",
            "Adjuntar antecedentes disponibles (si aplica)",
        ],
        "inputs": [],
        "validacion": "¿Se notificó la necesidad de devolución a compras?",
        "next": "T9_notificar_proveedor",
    },

    "T9_notificar_proveedor": {
        "type": "task",
        "titulo": "Notificar proveedor",
        "rol": "Ingeniero de abastecimiento",
        "descripcion": "Informar al proveedor sobre la devolución requerida para coordinar las siguientes acciones.",
        "checklist": [
            "Notificar al proveedor",
            "Registrar evidencia de la notificación",
        ],
        "inputs": [],
        "validacion": "¿Se notificó al proveedor?",
        "next": "T10_notificar_almacenamiento",
    },

    "T10_notificar_almacenamiento": {
        "type": "task",
        "titulo": "Notificar a almacenamiento",
        "rol": "Ingeniero de abastecimiento",
        "descripcion": "Informar a almacenamiento que la devolución debe seguir su curso operativo.",
        "checklist": [
            "Notificar a almacenamiento",
            "Entregar antecedentes necesarios para continuar",
        ],
        "inputs": [],
        "validacion": "¿Se notificó a almacenamiento?",
        "next": "T11_crear_pedido",
    },

    "T11_crear_pedido": {
        "type": "task",
        "titulo": "Crear pedido devolución",
        "rol": "Especialista de almacenamiento (solicitante)",
        "descripcion": "Crear el pedido de devolución correspondiente para habilitar la salida de entrega.",
        "checklist": [
            "Crear pedido de devolución",
            "Verificar que el pedido quede correctamente registrado",
        ],
        "inputs": [
            {"key":"pedido_devolucion", "label":"Agrega", "required": True},
        ],
        "validacion": "¿Se creó el pedido de devolución?",
        "next": "T12_crear_salida",
    },

    "T12_crear_salida": {
        "type": "task",
        "titulo": "Crear salida de entrega",
        "rol": "Especialista de almacenamiento (solicitante)",
        "descripcion": "Generar la salida de entrega del material devuelto en el sistema.",
        "checklist": [
            "Crear salida de entrega",
        ],
        "inputs": [],
        "validacion": "¿Se creó la salida de entrega?",
        "next": "T13_generar_picking",
    },

    "T13_generar_picking": {
        "type": "task",
        "titulo": "Generar picking",
        "rol": "Especialista de almacenamiento (solicitante)",
        "descripcion": "Generar el picking asociado a la devolución para preparar el despacho al proveedor.",
        "checklist": [
            "Generar picking",
        ],
        "inputs": [],
        "validacion": "¿Se generó el picking?",
        "next": "T14_informar_proveedor",
    },

    "T14_informar_proveedor": {
        "type": "task",
        "titulo": "Informar a proveedor",
        "rol": "Especialista de almacén",
        "descripcion": "Informar al proveedor los antecedentes de la devolución y su preparación.",
        "checklist": [
            "Informar al proveedor",
            "Entregar antecedentes de la devolución",
        ],
        "inputs": [],
        "validacion": "¿Se informó al proveedor?",
        "next": "D2_confirmacion_recepcion",
    },

    "D2_confirmacion_recepcion": {
        "type": "decision",
        "titulo": "Validación — Recepción de información por parte del proveedor",
        "rol": "Ingeniero de abastecimiento",
        "descripcion": "Verificar si el proveedor recepcionó la información enviada para continuar con el proceso.",
        "pregunta": "¿Se valida que el proveedor haya recepcionado la información?",
        "opciones": [
            {"label": "SÍ", "next": "T15_revision_facturas"},
            {"label": "NO", "next": "T14_informar_proveedor"},
        ],
        "ayuda": "Si no existe confirmación, se vuelve a informar al proveedor.",
    },

    "T15_revision_facturas": {
        "type": "task",
        "titulo": "Revisión y registro de facturas",
        "rol": "Ingeniero de abastecimiento",
        "descripcion": "Revisar y registrar las facturas relacionadas con la devolución para continuar con el cierre financiero.",
        "checklist": [
            "Revisar facturas asociadas",
            "Registrar facturas según corresponda",
        ],
        "inputs": [],
        "validacion": "¿Se revisaron y registraron las facturas?",
        "next": "T16_notificar_deposito",
    },

    "T16_notificar_deposito": {
        "type": "task",
        "titulo": "Notificar a proveedor necesidad de depósito",
        "rol": "Ingeniero de abastecimiento",
        "descripcion": "Informar al proveedor que debe realizar el depósito correspondiente como parte del proceso de regularización.",
        "checklist": [
            "Notificar al proveedor la necesidad de depósito",
            "Registrar la comunicación realizada",
        ],
        "inputs": [],
        "validacion": "¿Se notificó la necesidad de depósito?",
        "next": "T17_proveedor_deposito",
    },

    "T17_proveedor_deposito": {
        "type": "task",
        "titulo": "Proveedor realiza depósito",
        "rol": "Proveedor",
        "descripcion": "El proveedor realiza el depósito solicitado para finalizar el proceso de regularización.",
        "checklist": [
            "Proveedor realiza depósito",
            "Confirmar la recepción del depósito",
        ],
        "inputs": [],
        "validacion": "¿El proveedor realizó el depósito?",
        "next": "T18_compensar_nc",
    },

    "T18_compensar_nc": {
        "type": "task",
        "titulo": "Compensar nota de crédito",
        "rol": "Tesorero",
        "descripcion": "Compensar la nota de crédito o el depósito asociado para cerrar el proceso de devolución.",
        "checklist": [
            "Compensar nota de crédito",
            "Validar cierre financiero del caso",
        ],
        "inputs": [],
        "validacion": "¿Se compensó la nota de crédito?",
        "next": "END_RAMA_NO",
    },

    "END_RAMA_SI": {
        "type": "end",
        "titulo": "Proceso finalizado — Devolución inmediata",
        "rol": "HMI",
        "descripcion": "Se completó la ruta de devolución inmediata al proveedor.",
        "mensaje": "Fin de la rama de devolución inmediata.",
        "estado_final": FINALIZADO,
        "show_tables": True,
    },

    "END_RAMA_NO": {
        "type": "end",
        "titulo": "Proceso finalizado — Devolución con gestión posterior",
        "rol": "HMI",
        "descripcion": "Se completó la ruta de devolución posterior al proveedor.",
        "mensaje": "Fin de la rama de gestión posterior.",
        "estado_final": FINALIZADO,
        "show_tables": True,
    },

    "END_STOP": {
        "type": "end",
        "titulo": "Proceso detenido (STOP)",
        "rol": "HMI",
        "descripcion": "El proceso fue detenido por STOP.",
        "mensaje": "Fin por STOP.",
        "estado_final": DETENIDO_STOP,
        "show_tables": True,
    },
}


# ============================================================
# MOTOR HMI (NO EDITAR salvo mejoras estructurales)
# ============================================================

class HMIBase:
    def __init__(self):
        self.nodo_id = "D1_inmediata"
        self.estado = EN_CURSO
        self.run_id = str(uuid.uuid4())
        self.start_ts = _now_iso()
        self.end_ts = None

        self.inputs = {}
        self.decisiones = []  # SOLO rombos (y eventos STOP si aplica)
        self.logs = []
        self.historial = []

        self.output = widgets.Output()

        # Botones estilo PRO173/PRO174
        self.btn_si = widgets.Button(description="SÍ", button_style="success", layout={"width":"32%","height":"44px"})
        self.btn_no = widgets.Button(description="NO", button_style="danger", layout={"width":"32%","height":"44px"})
        self.btn_stop = widgets.Button(description="🛑 STOP", button_style="warning", layout={"width":"32%","height":"44px"})
        self.btn_volver = widgets.Button(description="⬅ Volver al paso anterior", layout={"width":"100%","height":"40px"})
        self.btn_exportar = widgets.Button(description="Exportar JSON (trazabilidad)", icon="download", layout={"width":"100%","height":"40px"})

        self.msg_box = widgets.HTML("")
        self._check_widgets = []
        self._decision_widget = None
        self._input_widgets = []  # list of (spec, widget)

        # STOP panel
        self.stop_panel = widgets.VBox([])
        self.sel_stop_motivos = widgets.SelectMultiple(options=MOTIVOS_STOP, layout=widgets.Layout(width="100%", height="120px"))
        self.txt_stop_detalle = widgets.Textarea(placeholder="Detalle (opcional)", layout=widgets.Layout(width="100%", height="70px"))
        self.btn_confirm_stop = widgets.Button(description="Confirmar STOP", button_style="warning", layout={"width":"100%","height":"40px"})
        self.btn_cancel_stop = widgets.Button(description="Cancelar STOP", layout={"width":"100%","height":"40px"})
        self._stop_open = False

        # Wire
        self.btn_si.on_click(self._on_si)
        self.btn_no.on_click(self._on_no)
        if ENABLE_STOP:
            self.btn_stop.on_click(self._on_stop_open)
            self.btn_confirm_stop.on_click(self._on_stop_confirm)
            self.btn_cancel_stop.on_click(self._on_stop_cancel)
        self.btn_volver.on_click(self._on_volver)
        self.btn_exportar.on_click(self._on_exportar)

        display(self.output)
        self.iniciar()

    def _log(self, tipo, data=None):
        self.logs.append({"ts": _now_iso(), "tipo": tipo, "nodo": self.nodo_id, "data": data or {}})

    def _push_history(self):
        self.historial.append(self.nodo_id)

    def _pop_history(self):
        return self.historial.pop() if self.historial else None

    def _clear_msg(self):
        self.msg_box.value = ""

    def _msg(self, text, kind="warn"):
        if kind == "ok":
            self.msg_box.value = (
                "<div style='margin-top:10px;padding:12px;border-radius:10px;background:#dcfce7;"
                "border:1px solid #22c55e;color:#14532d;'><b>%s</b></div>" % text
            )
        else:
            self.msg_box.value = (
                "<div style='margin-top:10px;padding:12px;border-radius:10px;background:#fee2e2;"
                "border:1px solid #ef4444;color:#7f1d1d;'><b>%s</b></div>" % text
            )

    def _render_header(self, n):
        badge = (
            "<span style='display:inline-block;padding:4px 10px;border-radius:999px;background:#eef2ff;"
            "border:1px solid #c7d2fe;font-size:12px;color:black;'><b>ROL:</b> %s</span>"
            % (n.get("rol",""))
        )
        return widgets.HTML(f"""
        <div style="padding:16px;border-radius:12px;background:#f8fafc;border:1px solid #e2e8f0;">
            <div style="font-size:12px;color:#0f172a;"><b>{PRO_ID}</b> — {PRO_TITULO}</div>
            <div style="margin-top:6px;font-size:22px;color:#0f172a;"><b>{n.get('titulo','')}</b></div>
            <div style="margin-top:8px;">{badge}</div>
            <div style="margin-top:10px;color:#0f172a;font-size:14px;line-height:1.35;white-space:pre-wrap;">{n.get('descripcion','')}</div>
        </div>
        """)

    def _render_task(self, n):
        valid = n.get("validacion","")

        # Checklist
        self._check_widgets = [
            widgets.Checkbox(description=item, value=False, layout=widgets.Layout(width="100%"))
            for item in (n.get("checklist",[]) or [])
        ]

        # Inputs
        self._input_widgets = []
        input_box_children = []
        for spec in (n.get("inputs") or []):
            label = spec.get("label", spec.get("key","Campo"))
            multiline = bool(spec.get("multiline", False))
            if multiline:
                w = widgets.Textarea(
                    description=label + ":",
                    layout=widgets.Layout(width="100%", height="90px"),
                    style={"description_width":"initial"}
                )
            else:
                w = widgets.Text(
                    description=label + ":",
                    layout=widgets.Layout(width="100%"),
                    style={"description_width":"initial"}
                )
            key = spec.get("key")
            if key in self.inputs:
                w.value = self.inputs.get(key,"")
            self._input_widgets.append((spec, w))
            input_box_children.append(w)

        inputs_box = widgets.VBox([])
        if input_box_children:
            inputs_box = widgets.VBox([
                widgets.HTML("""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>✍️ INPUTS</b></div>
                </div>
                """),
                widgets.VBox(input_box_children)
            ])

        checklist_box = widgets.VBox([])
        if self._check_widgets:
            checklist_box = widgets.VBox([
                widgets.HTML("""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>🧾 ACCIONES (Checklist)</b></div>
                    <div style="margin-top:6px;font-size:12px;color:#0f172a;opacity:0.9;">
                        Ítems que contengan <b>“si aplica”</b> no son obligatorios para avanzar.
                    </div>
                </div>
                """),
                widgets.VBox(self._check_widgets)
            ])

        valid_box = widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #0ea5e9;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>✅ VALIDACIÓN</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{valid}</b></div>
                <div style="margin-top:6px;font-size:12px;color:#0f172a;">
                    Presiona <b>SÍ</b> para avanzar. Presiona <b>NO</b> para bloquear el paso (modo revisión).
                </div>
            </div>
        """)

        return widgets.VBox([inputs_box, checklist_box, valid_box])

    def _render_decision(self, n):
        opts = n.get("opciones",[])
        radios = widgets.RadioButtons(
            options=[(o["label"], o["next"]) for o in opts],
            layout={"width":"100%"},
            style={"description_width":"initial"},
        )
        help_txt = n.get("ayuda","")
        help_html = f"<div style='margin-top:10px;font-size:12px;color:#0f172a;opacity:0.9;'><b>Nota:</b> {help_txt}</div>" if help_txt else ""
        return widgets.VBox([
            widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>🔶 DECISIÓN (rombo)</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{n.get('pregunta','')}</b></div>
                {help_html}
            </div>
            """),
            radios
        ]), radios

    def _render_end(self, n):
        return widgets.HTML(f"""
        <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
            <div style="font-size:14px;color:#0f172a;"><b>{n.get('mensaje','')}</b></div>
        </div>
        """)

    def _render_stop_panel(self):
        if not ENABLE_STOP:
            self.stop_panel.children = []
            return
        if not self._stop_open:
            self.stop_panel.children = []
            return
        self.stop_panel.children = [
            widgets.HTML("""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #f59e0b;background:#fffbeb;">
                <div style="font-size:14px;color:#0f172a;"><b>🛑 STOP</b> — Seleccione motivo(s) y detalle (opcional).</div>
            </div>
            """),
            self.sel_stop_motivos,
            self.txt_stop_detalle,
            widgets.HBox([self.btn_confirm_stop, self.btn_cancel_stop], layout=widgets.Layout(gap="8px"))
        ]

    def _render_tables_if_end(self, n):
        if n.get("type") != "end" or not n.get("show_tables", False):
            return widgets.VBox([])

        inputs_rows = "".join([f"<tr><td><b>{k}</b></td><td>{(v or '')}</td></tr>" for k, v in self.inputs.items()])
        if not inputs_rows:
            inputs_rows = "<tr><td colspan='2'>(sin inputs)</td></tr>"

        dec_rows = "".join([
            f"<tr><td>{d.get('ts','')}</td><td>{d.get('nodo','')}</td><td>{d.get('seleccion','')}</td></tr>"
            for d in self.decisiones
        ])
        if not dec_rows:
            dec_rows = "<tr><td colspan='3'>(sin decisiones)</td></tr>"

        return widgets.VBox([
            widgets.HTML(f"""
            <div style="margin-top:10px;padding:12px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:12px;color:#0f172a;"><b>📌 INPUTS</b></div>
                <table style="width:100%;border-collapse:collapse;margin-top:8px;font-size:12px;">
                    <thead>
                      <tr>
                        <th style="border:1px solid #e2e8f0;padding:5px;text-align:left;">Campo</th>
                        <th style="border:1px solid #e2e8f0;padding:5px;text-align:left;">Valor</th>
                      </tr>
                    </thead>
                    <tbody>{inputs_rows}</tbody>
                </table>
            </div>
            """),
            widgets.HTML(f"""
            <div style="margin-top:10px;padding:12px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:12px;color:#0f172a;"><b>🧭 DECISIONES (rombos)</b></div>
                <table style="width:100%;border-collapse:collapse;margin-top:8px;font-size:12px;">
                    <thead>
                      <tr>
                        <th style="border:1px solid #e2e8f0;padding:5px;text-align:left;">Timestamp</th>
                        <th style="border:1px solid #e2e8f0;padding:5px;text-align:left;">Nodo</th>
                        <th style="border:1px solid #e2e8f0;padding:5px;text-align:left;">Selección</th>
                      </tr>
                    </thead>
                    <tbody>{dec_rows}</tbody>
                </table>
            </div>
            """),
        ])

    def _render_footer(self):
        if ENABLE_STOP:
            top_row = widgets.HBox(
                [self.btn_si, self.btn_no, self.btn_stop],
                layout=widgets.Layout(justify_content="space-between", gap="8px", margin="10px 0")
            )
        else:
            self.btn_si.layout.width = "49%"
            self.btn_no.layout.width = "49%"
            top_row = widgets.HBox(
                [self.btn_si, self.btn_no],
                layout=widgets.Layout(justify_content="space-between", gap="8px", margin="10px 0")
            )

        return widgets.VBox([
            top_row,
            self.btn_volver,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.btn_exportar,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.stop_panel,
            self.msg_box,
        ])

    # ---------------- checks ----------------

    def _collect_inputs_required(self):
        n = NODOS[self.nodo_id]
        for spec, w in self._input_widgets:
            key = spec.get("key")
            required = bool(spec.get("required", False))
            val = (w.value or "").strip()
            if required and not val:
                return False, f"Debe completar el campo obligatorio: {spec.get('label', key)}"
            if key:
                self.inputs[key] = val
        return True, ""

    def _checklist_obligatorio_ok(self):
        oblig = [cb for cb in self._check_widgets if not _es_opcional_por_si_aplica(cb.description)]
        return all(cb.value for cb in oblig) if oblig else True

    # ---------------- render ----------------

    def iniciar(self):
        self._render()

    def _render(self):
        with self.output:
            clear_output(wait=True)
            self._clear_msg()

            n = NODOS[self.nodo_id]
            header = self._render_header(n)

            if n["type"] == "task":
                body = self._render_task(n)
                self._decision_widget = None
            elif n["type"] == "decision":
                body, radios = self._render_decision(n)
                self._decision_widget = radios
                self._check_widgets = []
                self._input_widgets = []
            else:
                body = self._render_end(n)
                self._decision_widget = None
                self._check_widgets = []
                self._input_widgets = []

            self._render_stop_panel()
            end_tables = self._render_tables_if_end(n)
            footer = self._render_footer()

            display(widgets.VBox([header, body, end_tables, footer]))

    # ---------------- events ----------------

    def _on_si(self, _):
        n = NODOS[self.nodo_id]
        if n["type"] == "end":
            self._msg("Este es un nodo final.", "ok")
            return

        if n["type"] == "decision":
            if self._decision_widget is None or self._decision_widget.value is None:
                self._msg("Debe seleccionar una opción.", "warn")
                self._log("VALIDACION_FALLA", {"mensaje": "sin selección"})
                return

            label_map = dict(self._decision_widget.options)
            self.decisiones.append({
                "ts": _now_iso(),
                "nodo": self.nodo_id,
                "seleccion": f"{label_map.get(self._decision_widget.value, str(self._decision_widget.value))}"
            })

            nxt = self._decision_widget.value
            self._push_history()
            self._log("AVANZA", {"next": nxt})
            self.nodo_id = nxt

            if NODOS[self.nodo_id]["type"] == "end":
                self.estado = NODOS[self.nodo_id].get("estado_final", FINALIZADO)
                self.end_ts = _now_iso()

            self._render()
            return

        if not self._checklist_obligatorio_ok():
            self._msg("Debe completar las acciones obligatorias antes de avanzar.", "warn")
            self._log("VALIDACION_FALLA", {"mensaje": "checklist obligatorio incompleto"})
            return

        ok_inputs, msg_inputs = self._collect_inputs_required()
        if not ok_inputs:
            self._msg(msg_inputs, "warn")
            self._log("VALIDACION_FALLA", {"mensaje": msg_inputs})
            return

        self._push_history()
        nxt = n.get("next")
        self._log("AVANZA", {"next": nxt})
        self.nodo_id = nxt

        if NODOS[self.nodo_id]["type"] == "end":
            self.estado = NODOS[self.nodo_id].get("estado_final", FINALIZADO)
            self.end_ts = _now_iso()

        self._render()

    def _on_no(self, _):
        n = NODOS[self.nodo_id]
        if n["type"] == "decision":
            self._msg("En decisiones: seleccione opción en el rombo y luego presione SÍ.", "warn")
            return
        if n["type"] == "end":
            self._msg("Este es un nodo final.", "ok")
            return
        self.estado = BLOQUEADO
        self._log("BLOQUEA", {"motivo": "Respuesta NO en validación"})
        self._msg("Paso bloqueado: respondió NO en la validación.", "warn")
        self._render()

    def _on_volver(self, _):
        prev = self._pop_history()
        if prev is None:
            self._msg("No hay paso anterior.", "warn")
            return
        self.nodo_id = prev
        self.estado = EN_CURSO
        self._log("VOLVER", {"to": prev})
        self._render()

    def _on_stop_open(self, _):
        self._stop_open = True
        self._render()

    def _on_stop_cancel(self, _):
        self._stop_open = False
        self.sel_stop_motivos.value = ()
        self.txt_stop_detalle.value = ""
        self._render()

    def _on_stop_confirm(self, _):
        motivos = list(self.sel_stop_motivos.value)
        detalle = (self.txt_stop_detalle.value or "").strip()
        self._log("STOP", {"motivos": motivos, "detalle": detalle})

        # Guardar evento STOP como "decisión" solo si quieres auditarlo (opcional)
        self.decisiones.append({
            "ts": _now_iso(),
            "nodo": self.nodo_id,
            "seleccion": f"STOP: {motivos} | {detalle}"
        })

        self.estado = DETENIDO_STOP
        self.end_ts = _now_iso()
        self._push_history()
        self.nodo_id = "END_STOP"
        self._stop_open = False
        self._render()

    def _on_exportar(self, _):
        payload = {
            "proceso": f"{PRO_ID} — {PRO_TITULO}",
            "run_id": self.run_id,
            "estado": self.estado,
            "start_ts": self.start_ts,
            "end_ts": self.end_ts,
            "current_node": self.nodo_id,
            "history_stack": list(self.historial),
            "decisiones": list(self.decisiones),  # SOLO rombos (+ STOP si lo guardas)
            "inputs": dict(self.inputs),
            "logs": list(self.logs),
            "export_ts": _now_iso(),
        }
        pretty = json.dumps(payload, ensure_ascii=False, indent=2)
        self.msg_box.value = f"""
        <div style='margin-top:10px;padding:12px;border-radius:10px;background:#dcfce7;border:1px solid #22c55e;color:#14532d;'>
            <b>📦 Export JSON (trazabilidad)</b>
            <pre style='white-space:pre-wrap;margin-top:10px;color:#14532d;'>{pretty}</pre>
        </div>
        """

# Ejecutar HMI
hmi = HMIBase()

hmi.iniciar()

Output()